In [25]:
import torch
import re
from collections import defaultdict



In [26]:
text = open("train.txt", "r", encoding="utf-8").read()

unique_chars = sorted(list(set(text)))
vocab_size = len(unique_chars)


print(''.join(unique_chars))
print(f"\nvocab_Size:{vocab_size}")

# unique_chars.extend(['4','6','7','9','0'])
# Adding remaining numbers 

def word_tokenize(text):
    """Each word is a token, but also each special character, spaces arent ocnsidered."""
    # Split on spaces but keep punctuation as separate tokens
    tokens = re.findall(r"\w+|[^\w\s]", text)
    return tokens
word_tokens = word_tokenize(text)


print(f"\nFirst 50 WORD tokens:\n {word_tokens[:50]}")




 !"&'(),-./12358:;?ABCDEFGHIJKLMNOPQRSTUVWYabcdefghijklmnopqrstuvwxyzé

vocab_Size:71

First 50 WORD tokens:
 ['A', 'SCANDAL', 'IN', 'BOHEMIA', 'Arthur', 'Conan', 'Doyle', 'Table', 'of', 'contents', 'Chapter', '1', 'Chapter', '2', 'Chapter', '3', 'CHAPTER', 'I', 'To', 'Sherlock', 'Holmes', 'she', 'is', 'always', 'the', 'woman', '.', 'I', 'have', 'seldom', 'heard', 'him', 'mention', 'her', 'under', 'any', 'other', 'name', '.', 'In', 'his', 'eyes', 'she', 'eclipses', 'and', 'predominates', 'the', 'whole', 'of', 'her']


In [27]:
def get_vocab(text):
    """Convert words into character seq. with END marker for Bytepairencoding"""
    vocab = defaultdict(int)
    for word in text.strip().split():
        # 'test' becomes ('t','e','s','t','<end>')
        vocab[' '.join(list(word)) + ' <end>'] += 1
    return vocab

def get_stats(vocab):
    """Count frequency of every adjacent pair and stores in dict pairs"""
    pairs = defaultdict(int)
    for word, freq in vocab.items():
        symbols = word.split()
        for i in range(len(symbols) - 1):
            pairs[symbols[i], symbols[i+1]] += freq
    return pairs

def merge_mostfrequent(best_pair, vocab):
    """Merge the most frequent pair across all words, variable- pair is the most freq adjacent pair"""
    new_vocab = {}
    to_replace = ' '.join(best_pair)
    replacement = ''.join(best_pair)
    for word in vocab:
        new_word = word.replace(to_replace, replacement)
        new_vocab[new_word] = vocab[word]
    return new_vocab


def train_bpe(text, n_merges):
    vocab = get_vocab(text)
    merges = []

    for i in range(n_merges):
        pairs = get_stats(vocab)
        if not pairs:
            break
        # merge the most frequent pair
        best_pair = max(pairs, key=pairs.get)
        vocab = merge_mostfrequent(best_pair, vocab)
        merges.append(best_pair)

    return vocab, merges

print("\n\n")
bpe_vocab, merges = train_bpe(text, n_merges=100)

# Build final token list from BPE vocab
bpe_tokens = set()
for word in bpe_vocab:
    for token in word.split():
        bpe_tokens.add(token)

bpe_tokens = sorted(list(bpe_tokens))
print(f"BPE vocab size: {len(bpe_tokens)}")
print(f"First 20 tokens: {bpe_tokens[:20]}")

stoi = {token: i for i, token in enumerate(bpe_tokens)}
itos = {i: token for i, token in enumerate(bpe_tokens)}

bpe_vocab_size=len(bpe_tokens)
#string ot integer and integer to string

def encode(text):
    words = text.strip().split()
    token_ids = []
    for word in words:
        # split word into characters with end marker
        symbols = list(word) + ['<end>']
        # apply each merge in order
        for pair in merges:
            i = 0
            while i < len(symbols) - 1:
                if symbols[i] == pair[0] and symbols[i+1] == pair[1]:
                    symbols = symbols[:i] + [''.join(pair)] + symbols[i+2:]
                else:
                    i += 1
        # look up each resulting token in stoi
        for token in symbols:
            if token in stoi:
                token_ids.append(stoi[token])
    return token_ids

encoded = encode(text)

def decode(token_ids):
    tokens = [itos[i] for i in token_ids]
    text = ''.join(tokens)           # no spaces between tokens
    text = text.replace('<end>', ' ') # <end> marks word boundaries → space
    return text.strip()

n = int(0.9 * len(encoded))
train = encoded[:n]
test = encoded[n:]


train = torch.tensor(train, dtype=torch.long)
test = torch.tensor(test, dtype=torch.long)

print(f"Train tokens: {len(train)}, Test tokens: {len(test)}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")






BPE vocab size: 469
First 20 tokens: ['!', '"', '"<end>', '&', "'", '(', ')', ',', ',<end>', '-', '.', '."<end>', '.<end>', '/', '1', '2', '3', '5', '8', ':']
Train tokens: 24012, Test tokens: 2668
Device: CPU


In [28]:
torch.manual_seed(10)
block_size = 8   # context length 
batch_size = 4   #no of sampleing chunks in one forward pass

def get_batch(data):
    #random start positions
    ix = torch.randint(len(data) - block_size, (batch_size,))
    
    # x is the input context, y is the target (shifted by 1)
    x = torch.stack([data[i : i+block_size] for i in ix])
    y = torch.stack([data[i+1 : i+block_size+1] for i in ix])
    
    return x, y

xb, yb = get_batch(train)
print(xb.shape)
print(yb.shape,"\n\n")

print("INPUTS",xb)
print("OUTPUTS:",yb)

    

torch.Size([4, 8])
torch.Size([4, 8]) 


INPUTS tensor([[233,  86, 133, 152,  49, 389, 304, 316],
        [133, 151, 100, 154, 416, 155, 388, 159],
        [215, 149, 265, 151, 461, 151,   8, 203],
        [441, 296, 232,  49, 389,  25, 329, 291]])
OUTPUTS: tensor([[ 86, 133, 152,  49, 389, 304, 316,  26],
        [151, 100, 154, 416, 155, 388, 159, 215],
        [149, 265, 151, 461, 151,   8, 203, 441],
        [296, 232,  49, 389,  25, 329, 291, 462]])


In [29]:
torch.manual_seed(10)
import torch.nn as nn  

import torch.nn.functional as F

class BigramLM(nn.Module):
    
    def __init__(self, vocab_size):
        super().__init__()
        #embedding table. or lookup table.
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self,index,targets=None):
        logits=self.token_embedding_table(index)
        #cross entropy needs a B C T not B T C matrix so we reshape.
        if targets is None:
            loss=None
        else:
            B,T,C=logits.shape
            logits=logits.view(B*T,C) #stretches the original 4x8 array into 1d
            targets=targets.view(B*T) #stretches the original 4x8 array into 1d
            loss=F.cross_entropy(logits,targets)

        return logits,loss

    def generate(self, index, max_new_tokens):
        #index is B x T array of indices in the current context,.
        for i in range(max_new_tokens):

            logits, loss = self.forward(index,None) 
            # gives B T C , we need last one.,
            logits = logits[:, -1, :]  # shape (B, C)

            probs = F.softmax(logits, dim=-1)  # shape (B, C)

            next_token = torch.multinomial(probs, num_samples=1)  # shape (B, 1)

            #pppend ,repeat
            index = torch.cat([index, next_token], dim=1) 

        return index
    
model=BigramLM(bpe_vocab_size)
logits,loss=model(xb,yb)
print(logits.shape)
print(loss)   #lideally loss should be -ln(1/bpe_vocab_size)
#output is a PREDICTION score for EACH element in the torch.stack 2d array, for each unique token, it gives a UN-NORMALISED probability that thats the next token


torch.Size([32, 469])
tensor(6.5791, grad_fn=<NllLossBackward0>)


In [30]:
context = torch.zeros((1, 1), dtype=torch.long)
generated = model.generate(context, max_new_tokens=20)
print(decode(generated[0].tolist()))   # random because model isnt trained, and generate function puts random shit

!allelingwhatagafshedOonoyal&ly erou"hall Uher erou1


In [51]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
batch_size = 32
best_loss = 2.7893   #got 2.7893 from a lot of iterations.

for i in range(100):
    xb, yb = get_batch(train)
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if loss.item() < best_loss:
        best_loss = loss.item()
        torch.save(model.state_dict(), 'bigram_weights.pth')

    if i % 20 == 0:
        print(f"loss {loss.item():.4f}, min is {best_loss:.4f}")

print(f"Training done. Best loss: {best_loss:.4f}")

loss 2.9528, min is 2.7893
loss 3.1652, min is 2.7893
loss 3.0184, min is 2.7893
loss 3.0801, min is 2.7893
loss 3.1117, min is 2.7893
Training done. Best loss: 2.7893


In [75]:
model.load_state_dict(torch.load('bigram_weights.pth'))

context = torch.tensor([encode("Holmes")], dtype=torch.long)

generated = model.generate(context, max_new_tokens=200)
print(decode(generated[0].tolist()))

Holmes lime thensere to guttly on." I o, blander. It ach which I was on King there of a y. It is rocltlamed fet I so warknot ked the ple sted ached Baintitten wostantabout is mastand in no, or ewere Theardes, which scor, and lie packing for the pace a maid as the les, you locant of ther. TEulow Wionces toostry she sted to Bopeen blair. It was maspered. Stweish the ledrod?" "St
